# Barrido de la supresión de duplicados en DETR

DETR y DINO evalúan hoy sin suprimir (`nms_iou=None` en `models.registry`, commit `a32330f`).
Esto mide qué cuesta y dónde está el punto bueno, barriendo las dos etapas de
`utils.boxes.suppress_nested`:

| parámetro | qué caza | apagada en |
|---|---|---|
| `nms_iou` | duplicados **desplazados** (`batched_nms`, por clase) | `1.0` |
| `iomin` | duplicados **anidados**, que el IoU clásico deja pasar | `1.0` |

`(1.0, 1.0)` es lo que corre hoy; `(0.3, 0.8)`, lo de antes del refactor. El umbral de score
no está fijo: se rebusca en `val` por max-F2 en cada celda y recién ahí se mide `test`.

```bash
./nms_sweep.sh detr_unfreeze_ts10                   # vuelca (una pasada de GPU) y barre la rejilla
NMS_SWEEP_RUN=detr_unfreeze_ts10 jupyter lab notebooks/nms_sweep.ipynb
```

Las tres figuras se guardan en `notebooks/figures/nms_sweep_<corrida>_{A,B,C}.png`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
RUN = os.environ.get("NMS_SWEEP_RUN", "detr_unfreeze_ts10")
DIR = Path(os.environ.get("NMS_SWEEP_DIR") or ROOT / "runs" / RUN)

FIGURES = ROOT / "notebooks" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

sweep = pd.read_csv(DIR / "nms_sweep.csv").sort_values(["iomin", "nms_iou"])
curves = pd.read_csv(DIR / "nms_sweep_curves.csv")

# Slots 1-3 de la paleta categórica validada y la rampa secuencial azul, en modo claro.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
RAMP = ["#86b6ef", "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#0d366b"]
SURFACE, INK, INK_2, INK_3, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#8a8983", "#c9c8c3"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "figure.dpi": 120,
    "font.size": 9, "axes.titlesize": 10, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "text.color": INK, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
    "legend.frameon": False,
})

IOMIN = sorted(sweep.iomin.unique())
NMS = sorted(sweep.nms_iou.unique())
COLOR = dict(zip(IOMIN, SERIES))     # el color sigue a la entidad, nunca al ranking
# La rejilla de `nms_iou` es desigual (0.1, 0.15, 0.2, ...): en un eje numérico los primeros
# puntos se pisan, así que se dibuja equiespaciada y el valor va al tick.
X = {v: i for i, v in enumerate(NMS)}
BASE = sweep[(sweep.nms_iou == max(NMS)) & (sweep.iomin == max(IOMIN))].squeeze()

def etiqueta(iomin):
    return "iomin apagado" if iomin >= 1.0 else f"iomin {iomin:g}"

def porcentaje(ax, eje="y"):
    """Un eje casi plano con formato `.0%` imprime ocho veces la misma etiqueta."""
    lim = ax.get_ylim() if eje == "y" else ax.get_xlim()
    decimales = 1 if lim[1] - lim[0] < 0.12 else 0
    (ax.yaxis if eje == "y" else ax.xaxis).set_major_formatter(lambda v, _: f"{v:.{decimales}%}")

def panel(ax, columna, titulo, ylabel, percent=False):
    finales = []
    for iomin in IOMIN:
        parte = sweep[sweep.iomin == iomin].sort_values("nms_iou")
        ax.plot([X[v] for v in parte.nms_iou], parte[columna], color=COLOR[iomin], linewidth=2,
                marker="o", markersize=6, markeredgecolor=SURFACE, markeredgewidth=1.5,
                clip_on=False, zorder=3)
        finales.append((float(parte.iloc[-1][columna]), etiqueta(iomin)))

    ax.axvline(X[0.3], color=AXIS, linewidth=1, linestyle=(0, (3, 3)), zorder=1)
    ax.set_title(titulo, loc="left", pad=8)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("nms_iou")
    ax.set_xticks(list(X.values()))
    ax.set_xticklabels([("1.0\napagado" if n >= 1.0 else f"{n:g}") for n in NMS])
    ax.set_xlim(-0.35, len(NMS) + 0.65)   # aire a la derecha para las etiquetas directas

    if percent:
        bajo, alto = ax.get_ylim()
        if alto - bajo < 0.05:            # una serie constante deja el eje sin rango
            ax.set_ylim((bajo + alto) / 2 - 0.025, (bajo + alto) / 2 + 0.025)
        porcentaje(ax)

    # Etiqueta directa, separando las que se pisarían: los tres slots claros no llegan a 3:1
    # contra la superficie, así que la identidad no puede quedar sólo en el color.
    bajo, alto = ax.get_ylim()
    y = bajo - 1
    for valor, texto in sorted(finales):
        y = max(valor, y + (alto - bajo) * 0.055)
        ax.annotate(texto, (len(NMS) - 0.82, y), va="center", fontsize=7.5, color=INK_2,
                    annotation_clip=False)

def figura(bloque, titulo, nota):
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.8))
    fig.nombre = f"nms_sweep_{RUN}_{bloque}.png"
    fig.subplots_adjust(top=0.845, bottom=0.08, left=0.07, right=0.97, hspace=0.5, wspace=0.34)
    fig.suptitle(titulo, x=0.008, y=0.985, ha="left", fontsize=12, color=INK)
    fig.legend(handles=[Line2D([], [], color=COLOR[i], linewidth=2, marker="o", markersize=6,
                               label=etiqueta(i)) for i in IOMIN],
               loc="upper left", bbox_to_anchor=(0.008, 0.945), ncol=len(IOMIN), fontsize=8,
               handlelength=1.6, columnspacing=1.4)
    fig.text(0.008, 0.895, nota, fontsize=8, color=INK_3)
    return fig, axes

def guardar(fig):
    path = FIGURES / fig.nombre
    fig.savefig(path, dpi=200, facecolor=SURFACE)
    print(path.relative_to(ROOT))

print(f"{RUN}: {len(sweep)} combinaciones | nms_iou {NMS} | iomin {IOMIN}")

## 0. La tabla

La vista tabular de todo lo que sigue, para leerlo sin depender del color.

In [ ]:
tabla = sweep.assign(supresion=[
    "ninguna" if n >= 1.0 and i >= 1.0 else f"nms {n:g} / iomin {i:g}"
    for n, i in zip(sweep.nms_iou, sweep.iomin)
]).set_index("supresion").drop(columns=["nms_iou", "iomin"])

tabla.style.format("{:.3f}").format({"cajas_por_ventana": "{:.1f}", "fp_per_hour": "{:.0f}"}) \
    .background_gradient(subset=["map_30", "f_beta"], cmap="Blues")

## A. Sin umbral de score

No dependen del punto de operación, así que aíslan lo único que importa: **si la supresión
borró duplicados o borró cajas buenas**. mAP que sube al suprimir = eran duplicados.

In [ ]:
fig, axes = figura("A", "A. Métricas independientes del umbral de score",
                   "mAP más alto al suprimir = eran duplicados. Más bajo = se borraron cajas buenas.")
panel(axes[0, 0], "map_30", "1. mAP@0.3 — el que elige `best.pt`", "mAP@0.3")
panel(axes[0, 1], "map_50", "2. mAP@0.5", "mAP@0.5")
panel(axes[1, 0], "map_50_95", "3. mAP@0.5:0.95", "mAP@0.5:0.95")
panel(axes[1, 1], "cajas_por_ventana", "4. Cajas por ventana en test", "cajas / ventana")
guardar(fig)
plt.show()

## B. En el punto de operación

Umbral elegido en `val` por max-F2, medido una sola vez en `test`. Como se rebusca en cada
celda, no confunden la supresión con un corrimiento del punto de operación.

In [ ]:
fig, axes = figura("B", "B. En el umbral max-F2 elegido en val, medido en test",
                   "F2 pesa el recall cuatro veces más que la precisión: es el que decide.")
panel(axes[0, 0], "recall", "5. Recall — lo que se paga", "recall", percent=True)
panel(axes[0, 1], "precision", "6. Precisión — lo que se compra", "precisión", percent=True)
panel(axes[1, 0], "f_beta", "7. F2 — el criterio de la tesis", "F2")
panel(axes[1, 1], "fp_per_hour", "8. Falsos positivos por hora", "FP / h")
guardar(fig)
plt.show()

## C. Diagnóstico

Por qué se mueve lo de arriba. El panel 11 usa una rampa de un solo tono porque `nms_iou` es
una magnitud ordenada, no una identidad.

In [ ]:
IOMIN_CURVAS = 0.8 if 0.8 in IOMIN else IOMIN[0]

fig, axes = figura("C", "C. De dónde sale la diferencia",
                   "El número junto a cada punto del panel 12 es su nms_iou.")
panel(axes[0, 0], "umbral", "9. Umbral de score elegido en val", "umbral")
panel(axes[0, 1], "recall_techo", "10. Recall máximo con precisión ≥ 0.70", "recall", percent=True)

# 11. Curva PR completa para un solo `iomin`: tres familias encimadas no se leen.
ax = axes[1, 0]
paso = max(1, len(RAMP) // len(NMS))
for i, nms_iou in enumerate(NMS):
    curva = curves[(curves.iomin == IOMIN_CURVAS) & (curves.nms_iou == nms_iou)]
    ax.plot(curva.recall, curva.precision, color=RAMP[min(i * paso, len(RAMP) - 1)],
            linewidth=2, label=f"{nms_iou:g}", zorder=3)
ax.set_title(f"11. Precisión-recall barriendo el score ({etiqueta(IOMIN_CURVAS)})", loc="left", pad=8)
ax.set_xlabel("recall")
ax.set_ylabel("precisión")
porcentaje(ax, "x")
porcentaje(ax, "y")
# Abajo a la izquierda: en una curva PR esa esquina está vacía, así que no tapa datos.
ax.legend(title="nms_iou", fontsize=7.5, title_fontsize=7.5, loc="lower left", ncol=2)

# 12. Cada combinación, un punto; el baseline sin suprimir va anillado.
ax = axes[1, 1]
marcados = {min(NMS), max(NMS), 0.3}
for iomin in IOMIN:
    parte = sweep[sweep.iomin == iomin].sort_values("nms_iou")
    ax.plot(parte.precision, parte.recall, color=COLOR[iomin], linewidth=1.2, alpha=0.55, zorder=2)
    ax.scatter(parte.precision, parte.recall, s=42, color=COLOR[iomin], edgecolor=SURFACE,
               linewidth=1.5, zorder=3)
    # Selectivas: los extremos y el default histórico. Un número sobre cada punto se vuelve
    # ilegible en cuanto la nube se junta.
    for _, fila in parte[parte.nms_iou.isin(marcados)].iterrows():
        ax.annotate(f"{fila.nms_iou:g}", (fila.precision, fila.recall), ha="center",
                    textcoords="offset points", xytext=(0, 9), fontsize=7, color=INK_3)
ax.scatter([BASE.precision], [BASE.recall], s=170, facecolor="none", edgecolor=INK_2,
           linewidth=1.4, zorder=4)
ax.annotate("sin suprimir\n(hoy)", (BASE.precision, BASE.recall), ha="center",
            textcoords="offset points", xytext=(0, -26), fontsize=7.5, color=INK_2)
ax.set_title("12. Frontera precisión-recall (cada punto, un nms_iou)", loc="left", pad=8)
ax.set_xlabel("precisión")
ax.set_ylabel("recall")
porcentaje(ax, "x")
porcentaje(ax, "y")
guardar(fig)
plt.show()

## Cómo leerlo

- **Panel 1**: si el mAP@0.3 sube al suprimir, los duplicados son reales y quitarle el NMS a
  DETR fue una regresión — hay que devolverle un `nms_iou` a `ARCHITECTURES` en
  `src/models/registry.py`.
- **Panel 12**: busca el punto que gane precisión sin caer por debajo del anillo en recall. Si
  toda la curva de un `iomin` queda abajo, esa etapa borra cajas legítimas.
- **Paneles 5 y 6 se leen con el 7**: F2 pesa el recall cuatro veces más, así que ganar 15 de
  precisión perdiendo 5 de recall puede seguir siendo peor.
- **Panel 4 es el control de cordura**: si las cajas por ventana casi no bajan, esa
  configuración no suprime nada y el resto es ruido.

Para la tesis, dos salvedades: el `best.pt` se eligió por mAP@0.3 **sin** suprimir, así que
esto es post-hoc sobre un checkpoint seleccionado bajo otra configuración; y YOLO trae NMS
interno de Ultralytics, así que esto es una ablación de DETR, no una comparación entre
arquitecturas.